#**Create local vector embeddings using sentens-transformer python library**

##**GOAL: to embed text sentences and perform semantic searches using your own Python code.**

There are many pre-trained embedding models available on Hugging Face that you can use to create vector embeddings.
Sentence Transformers (SBERT) is a library that makes it easy to use these models for vector embedding.

Use pip  to install  'sentence_transformers' library  and import  'SentenceTransformer model loader' from this library.

In [1]:
!pip install -q sentence-transformers

import os
import warnings

# hide download warnings and progress bars so the outputs stay readable
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer

Load the 'paraphrase-MiniLM-L6-v2' model  from HuggingFace resource  using the  SentenceTransformer( *model-name* )  and store the reference to the model object in the 'model' variable

In [2]:
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

After loading the model, call the 'encode()' method on the model object to create a vector representation of a specific text sentence. Use your own text string  as the parameter.

In [3]:
sentence = "The weather is really nice today, so we are going to have a picnic in the park."
embedding = model.encode(sentence)
print(embedding.shape)  # one vector with 384 numbers
embedding

(384,)


array([-1.70768350e-02, -3.76621890e-03,  4.72908556e-01,  4.13726658e-01,
        9.50965345e-01,  1.64198279e-01,  2.72472054e-01, -2.57006139e-01,
       -3.88104543e-02, -3.18381451e-02, -4.14676517e-01, -3.36701006e-01,
       -3.05337198e-02,  3.26829463e-01,  2.62492120e-01,  1.55002639e-01,
        2.36955330e-01, -6.87072054e-02,  1.05667025e-01, -6.60924166e-02,
       -1.46156028e-01, -3.10939342e-01,  3.66118968e-01,  7.26347268e-02,
       -8.20109427e-01,  1.68189168e-01, -2.09332764e-01,  7.83404410e-02,
       -4.97155637e-02,  2.07505822e-01,  2.87221640e-01, -4.13571708e-02,
       -4.26344365e-01, -6.39171302e-02, -1.27176329e-01,  3.94646585e-01,
        1.68670624e-01, -5.84521532e-01, -1.67323530e-01,  2.00690180e-01,
       -1.76541969e-01, -1.46926269e-01,  4.50198025e-01,  1.06034085e-01,
       -3.06940258e-01,  7.60056555e-01, -2.48864770e-01,  2.44951993e-01,
        3.60756725e-01,  3.50430071e-01,  3.89433622e-01,  3.31894517e-01,
        1.99824683e-02, -

Create vector representations for several text sentences. Place the text strings in a list and use this list as an argument. Use 8-10 sentences of 20-25 words each.  Call the 'encode()' method on the model object with the list of sentences as an argument.

In [4]:
# different topics, with pairs about weather, food and sport so similar ones can be checked
sentences_list = [
    "Heavy rain flooded several streets in the city center this morning, and many people were late for work because buses stopped running.",
    "The weather service warns that a strong storm with wind and thunder will reach the coast tonight, so people should stay at home.",
    "My grandmother makes the best pancakes on Sunday mornings, and she always serves them with fresh berries, honey and a big cup of tea.",
    "This new Italian restaurant near the station serves homemade pasta and pizza from a wood oven, and the prices are surprisingly low for downtown.",
    "Our football team won the final match of the season with a late goal, and the fans celebrated in the streets until midnight.",
    "She trained for almost a year to run her first marathon, and she finished the race in just under four hours last weekend.",
    "The new phone has a better camera and a bigger battery, but many users complain that it gets hot while playing games.",
    "Our dog learned to open the kitchen door by himself, so now we have to hide all the snacks on the top shelf.",
    "We spent two weeks travelling by train across Italy, visiting Rome, Florence and Venice, and eating gelato almost every single afternoon.",
    "Scientists found that sleeping less than six hours a night for a long time can raise the risk of heart problems.",
]

sent_emb = model.encode(sentences_list)
print(sent_emb.shape)  # 10 sentences, 384 numbers each

(10, 384)


#**Definition of semantic textual similaritye**

Import 'util' module from sentence_transformers library.

In [5]:
from sentence_transformers import util

You can calculate the cosine similarity of the vector representations of our sentences using the 'cos_sim()' function from the util module.
Example: sim = util.cos_sim(embedding_1, embedding_2). Calculate the cosine similarity for any two sentences from your list.


In [6]:
# rain and storm should be close, rain and dog should not
sim_weather = util.cos_sim(sent_emb[0], sent_emb[1])
sim_dog = util.cos_sim(sent_emb[0], sent_emb[7])

print("rain vs storm:", round(sim_weather.item(), 3))
print("rain vs dog:", round(sim_dog.item(), 3))

rain vs storm: 0.245
rain vs dog: -0.087


Write and test a function named 'cos_similarity_calculation' that determines the semantic similarity between the sentences in your list and any text sentence using their vector representations and the cosine distance as a similarity measure.  

In [7]:
def cos_similarity_calculation(text, sentences, top=3):
    text_emb = model.encode(text)
    all_emb = model.encode(sentences)
    scores = util.cos_sim(text_emb, all_emb)[0].tolist()

    # best matches first
    ranked = sorted(zip(scores, sentences), reverse=True)
    for score, s in ranked[:top]:
        print(round(score, 3), "|", s)
    return scores


question = "Where can I get some good food in town?"
scores = cos_similarity_calculation(question, sentences_list)

0.418 | This new Italian restaurant near the station serves homemade pasta and pizza from a wood oven, and the prices are surprisingly low for downtown.
0.381 | My grandmother makes the best pancakes on Sunday mornings, and she always serves them with fresh berries, honey and a big cup of tea.
0.24 | We spent two weeks travelling by train across Italy, visiting Rome, Florence and Venice, and eating gelato almost every single afternoon.


Create a function that determines the cosine similarity between a vector and a batch of vectors using the cosine distance formula and the numpy library. Add code to demonstrate how to use this function.

In [8]:
import numpy as np

def np_cos_sim(vec, batch):
    # dot product of the vector with every row, divided by the lengths
    dots = batch @ vec
    lengths = np.linalg.norm(batch, axis=1) * np.linalg.norm(vec)
    return dots / lengths


query_emb = model.encode("my phone battery dies too fast")
np_scores = np_cos_sim(query_emb, sent_emb)

best = np.argmax(np_scores)
print("best match:", sentences_list[best])
print("score:", round(float(np_scores[best]), 3))

# should give the same numbers as util.cos_sim
print(np.allclose(np_scores, util.cos_sim(query_emb, sent_emb)[0].numpy(), atol=1e-5))

best match: The new phone has a better camera and a bigger battery, but many users complain that it gets hot while playing games.
score: 0.511
True


### What I noticed

Each sentence becomes a vector of 384 numbers. Rain vs storm got 0.245 and rain vs dog got -0.087, so the model does see the topic, even if the score for two weather sentences is not very high. The function found the restaurant (0.418) and the pancakes for the food question, and the phone sentence for the battery question. My NumPy version gives the same numbers as `util.cos_sim`, so cosine similarity is just a dot product divided by the vector lengths.